# 🚀 TejaAI - High-Speed Model Server on Kaggle (Free T4 GPU)

This notebook loads your fine-tuned **`teja-ai-gemma4-e2b-t4-final`** adapter and serves an **OpenAI-compatible API** via a high-speed Cloudflare Tunnel.

### ⚠️ IMPORTANT KAGGLE SETTINGS (Right Sidebar):
1. **Accelerator**: Select **GPU T4 x 2** (or GPU T4)
2. **Internet**: Toggle **Internet ON** (Required to download packages and start Cloudflare tunnel!)
3. **Persistence**: Variables and Files (optional)

In [ ]:
# 2. Install required dependencies (Install Transformers with Gemma 4 support directly from GitHub)
!pip uninstall -y torchao
!pip install -q --upgrade "git+https://github.com/huggingface/transformers.git" "peft>=0.14.0" "accelerate>=1.2.0" fastapi uvicorn pydantic pyngrok


In [ ]:
# 3. Direct Hugging Face Adapter (No manual file uploading needed!)
ADAPTER_PATH = "Tulasiteja9/teja-ai-gemma4-e2b-lora"
print(f"✅ Cloud Adapter Repo: {ADAPTER_PATH}")


In [ ]:
# 4. Load Base Model and LoRA Adapter onto GPU
import os, torch
from transformers import AutoTokenizer
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

HF_TOKEN = os.getenv("HF_TOKEN", "") or "YOUR_HF_TOKEN_HERE"
BASE_MODEL_NAME = "google/gemma-4-E2B"

print('Loading tokenizer directly from Hugging Face...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading base model {BASE_MODEL_NAME}...')
dtype = torch.bfloat16 if device == 'cuda' else torch.float32
device_map = {'': 0} if device == 'cuda' else None

base_model = None
try:
    from transformers import AutoModelForImageTextToText
    base_model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    print('✅ Loaded via AutoModelForImageTextToText')
except Exception as e:
    print(f'Fallback to AutoModelForCausalLM: {e}')
    from transformers import AutoModelForCausalLM
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True,
        token=HF_TOKEN,
    )

print('Attaching fine-tuned LoRA adapter from Hugging Face...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, token=HF_TOKEN)
model.eval()
print('✅ SUCCESS: Fine-tuned Gemma 4 model is ready on GPU!')


In [ ]:
# 5. Create FastAPI OpenAI-Compatible Server & Start Background Thread
import time, threading, uvicorn, torch
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title='TejaAI Kaggle Model Server')

class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str = 'teja-ai-gemma4-e2b'
    messages: List[Message]
    temperature: Optional[float] = 0.2
    max_tokens: Optional[int] = 512

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'teja-ai-gemma4-e2b', 'device': device}

@app.post('/v1/chat/completions')
def chat_completions(req: ChatCompletionRequest):
    chat_msgs = [{'role': m.role, 'content': m.content} for m in req.messages]
    prompt = tokenizer.apply_chat_template(chat_msgs, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    prompt_len = inputs['input_ids'].shape[1]
    
    # Gemma 4 Stop Tokens (<turn|>, <eos>, <end_of_turn>)
    stop_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    for t_str in ['<turn|>', '<end_of_turn>', '<eos>']:
        tid = tokenizer.convert_tokens_to_ids(t_str)
        if tid is not None and tid != tokenizer.unk_token_id and tid not in stop_ids:
            stop_ids.append(tid)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=min(req.max_tokens or 256, 512),
            do_sample=req.temperature > 0.05,
            temperature=max(0.1, req.temperature),
            eos_token_id=stop_ids,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        
    new_tokens = out[0][prompt_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Remove trailing repeat artifacts if any
    for stop_str in ['<turn|>', '<end_of_turn>']:
        if stop_str in text:
            text = text.split(stop_str)[0].strip()
    
    return {
        'id': f'chatcmpl-{int(time.time())}',
        'object': 'chat.completion',
        'created': int(time.time()),
        'model': req.model,
        'choices': [
            {
                'index': 0,
                'message': {'role': 'assistant', 'content': text},
                'finish_reason': 'stop',
            }
        ],
        'usage': {
            'prompt_tokens': prompt_len,
            'completion_tokens': len(new_tokens),
            'total_tokens': prompt_len + len(new_tokens),
        },
    }


In [ ]:
# 5. Create FastAPI OpenAI-Compatible Server & Start Background Thread
import time, threading, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title='TejaAI Kaggle Model Server')

class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str = 'teja-ai-gemma4-e2b'
    messages: List[Message]
    temperature: Optional[float] = 0.2
    max_tokens: Optional[int] = 512

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'teja-ai-gemma4-e2b', 'device': device}

@app.post('/v1/chat/completions')
def chat_completions(req: ChatCompletionRequest):
    chat_msgs = [{'role': m.role, 'content': m.content} for m in req.messages]
    prompt = tokenizer.apply_chat_template(chat_msgs, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    prompt_len = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens or 256,
            do_sample=req.temperature > 0.05,
            temperature=max(0.1, req.temperature),
            pad_token_id=tokenizer.eos_token_id,
        )
        
    new_tokens = out[0][prompt_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    return {
        'id': f'chatcmpl-{int(time.time())}',
        'object': 'chat.completion',
        'created': int(time.time()),
        'model': req.model,
        'choices': [
            {
                'index': 0,
                'message': {'role': 'assistant', 'content': text},
                'finish_reason': 'stop',
            }
        ],
        'usage': {
            'prompt_tokens': prompt_len,
            'completion_tokens': len(new_tokens),
            'total_tokens': prompt_len + len(new_tokens),
        },
    }

In [ ]:
# 6. Clean Port 8000 & Launch Server + Cloudflare Tunnel
import subprocess, re, time

!fuser -k 8000/tcp 2>/dev/null || true
!pkill -9 -f cloudflared 2>/dev/null || true

def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print('✅ Local Uvicorn server started on port 8000.')

# Install Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print('Starting Cloudflare Tunnel...')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

tunnel_url = None
for _ in range(30):
    line = tunnel_process.stderr.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print('=' * 65)
    print('🎉 YOUR HIGH-SPEED KAGGLE GPU MODEL SERVER IS LIVE!')
    print(f'Public URL: {tunnel_url}/v1')
    print('=' * 65)
    print(f'\nAdd this to your local .env file:')
    print(f'TEJA_MODEL_URL={tunnel_url}/v1')
    print(f'ASTRA_MODEL=teja-gemma')
else:
    print('❌ Failed to obtain tunnel URL. Check if Internet is turned ON in Kaggle sidebar!')

# Keep notebook active
while True:
    time.sleep(30)